In [ ]:
import json
import pandas as pd

with open("train-v2.0.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Aplanar la estructura anidada de SQuAD 2.0 (data -> paragraphs -> qas)
rows = []
for article in data["data"]:
    title = article["title"]
    for paragraph in article["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            rows.append({
                "title": title,
                "context": context,
                "question": qa["question"],
                "id": qa["id"],
                "is_impossible": qa.get("is_impossible", False),
                "answers": qa["answers"],
            })

df = pd.DataFrame(rows)
df.head()


In [ ]:
df_por_contexto = (
    df.groupby("context")["question"]
    .apply(list)
    .reset_index(name="questions")
)
df_por_contexto


In [ ]:
df_por_contexto["num_questions"] = df_por_contexto["questions"].apply(len)
df_por_contexto["question_lens"] = df_por_contexto["questions"].apply(
    lambda qs: [len(q.split()) for q in qs]
)
df_por_contexto["avg_question_len"] = df_por_contexto["question_lens"].apply(
    lambda lens: sum(lens) / len(lens)
)
df_por_contexto
